# Imports

In [1]:
import re
import cda2
import math

import time
from datetime import datetime, timedelta

import pandas as pd
import pyarrow
from typing import Iterator, Tuple
from pyspark.sql.functions import pandas_udf

from pyspark.sql.window import Window
import pyspark.sql.types as T
import pyspark.sql.functions as F
from pyspark.sql.functions import col, row_number
from pyspark.sql.functions import explode, map_keys, col

In [2]:
api = cda2.Api()

In [3]:
# Set configuration parameters to better optimize queries.

config = {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize": "1m",
    "spark.executor.memory": "8g",
    "spark.executor.memoryOverhead": "16g",
}

In [4]:
# Start Spark and specify number of cpus to use.

api.start_spark(n_executors=100, config=config)

https://artifacts.mitre.org/artifactory/java-libs-release added as a remote repository with the name: repo-1
https://dali.mitre.org/nexus/content/repositories/mitre-caasd-releases added as a remote repository with the name: repo-2
https://dali.mitre.org/nexus/content/repositories/external-releases added as a remote repository with the name: repo-3
Ivy Default Cache set to: /home/rchong/.ivy2/cache
The jars for the packages stored in: /home/rchong/.ivy2/jars
org.mitre.spark#spark-geo_spark3.5_2.12 added as a dependency
org.apache.spark#spark-avro_2.12 added as a dependency
graphframes#graphframes added as a dependency
org.mongodb.spark#mongo-spark-connector_2.12 added as a dependency
com.oracle.database.jdbc#ojdbc8 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-16787223-1090-4e53-acfb-41686bd2482f;1.0
	confs: [default]


:: loading settings :: url = jar:file:/devel/data_access/software/tdp-jupyter/poetry/cache/virtualenvs/python39-QwwvzYkJ-py3.9/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.mitre.spark#spark-geo_spark3.5_2.12;0.2.0 in repo-1
	found org.mitre.spark#spark-geo-core_2.12;0.2.0 in repo-1
	found org.scala-lang.modules#scala-collection-compat_2.12;2.11.0 in central
	found net.sf.geographiclib#GeographicLib-Java;2.0 in central
	found org.ejml#ejml-core;0.43.1 in central
	found org.ejml#ejml-ddense;0.43.1 in central
	found com.esri.geometry#esri-geometry-api;2.2.4 in central
	found com.fasterxml.jackson.core#jackson-core;2.9.6 in central
	found com.google.geometry#s2-geometry;2.0.0 in central
	found com.google.guava#guava;25.1-jre in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found org.checkerframework#checker-qual;2.0.0 in central
	found com.google.errorprone#error_prone_annotations;2.1.3 in central
	found com.google.j2objc#j2objc-annotations;1.1 in central
	found org.codehaus.mojo#animal-sniffer-annotations;1.14 in central
	found com.uber#h3;4.1.1 in central
	found org.apache.spark#spark-avro_2.12;3.5.1 in central
	found org.tuka

In [5]:
year0 = "2025"
year1 = str(int(year0) + 1)

In [6]:
dates = {"start_date": year0 + "-01-01", "end_date": year1 +"-01-01"}

In [7]:
df_sectors = (
    api.dataframe("StaticEramAirspace",
                  **dates,
#                  partition_filters=api.custom_partitions(['ARTCC_SECTOR']),
#                  partition_filters=api.custom_partitions(['SECTOR']),
#                  partition_filters=api.custom_partitions(['SAA']),
#                  partition_filters=api.custom_partitions(['FAV']), //additional: {TYPE -> ENROUTE}
#                  partition_filters=api.custom_partitions(['ZDC']),
                  metadata=True)
    .select(
        F.col("metadata.custom_partitions").alias("partitions"),
        F.col("metadata.effective_end_date").alias("end_date"),
    )
)

Multiple versions found: 3.1.71, 3.1.73, 3.1.74, 3.1.76, 3.1.77, 3.1.79
                                                                                

In [8]:
df_sectors.count()

51579

In [9]:
#df_sectors.show(100, truncate=False)

In [10]:
#window = Window.partitionBy("grouping").orderBy(col("end_date").desc())
window = Window.partitionBy("partitions").orderBy(col("end_date").desc())

df_sectors_newest = (df_sectors
    .withColumn("row", row_number().over(window))
    .filter(col("row") == 1)
    .drop("row", "end_date")
    .orderBy("partitions")
)

In [11]:
df_sectors_newest.count()

100

In [12]:
df_sectors_newest.show()

+-------------------+
|         partitions|
+-------------------+
|[ZAB, ARTCC_SECTOR]|
|         [ZAB, FAV]|
|         [ZAB, SAA]|
|      [ZAB, SECTOR]|
|      [ZAB, TRACON]|
|[ZAU, ARTCC_SECTOR]|
|         [ZAU, FAV]|
|         [ZAU, SAA]|
|      [ZAU, SECTOR]|
|      [ZAU, TRACON]|
|[ZBW, ARTCC_SECTOR]|
|         [ZBW, FAV]|
|         [ZBW, SAA]|
|      [ZBW, SECTOR]|
|      [ZBW, TRACON]|
|[ZDC, ARTCC_SECTOR]|
|         [ZDC, FAV]|
|         [ZDC, SAA]|
|      [ZDC, SECTOR]|
|      [ZDC, TRACON]|
+-------------------+
only showing top 20 rows

